In [1]:
import numpy as np
import random 
import pandas as pd
import itertools

Moves:
first move only:
    double down - double the bet and take one more card
    split - seperate tow cards of the same calue into two hands
    surrender - forfeit the hand for half the bet back
    insurance - dealer face up card is ace but one-halg oringal bet that dealer has blackjack, pays 2 to 1
hit - take another cards
stand - take no more cards

Dealer 
hit until 17
hit soft 17 is a variation

In [ ]:
#random.seed(1)

sims = 1000000
deck_pen = 0.5

def game_sim(number_of_decks = 2,
             sims = 10000,
             deck_pen = 0.7,
             dealer_hit_soft_17 = True,
             double_allowed = True,
             split_allowed = True,
             surrender_allowed = True,
             insurance_allowed = True,
             blackjack_payout =('3:2', '6:5')
             ):
    
    # ---------------------------------------------
    # setting up decks
    cards = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']

    card_values = {
        # to do 
        #'A' : [1, 11],
        'A' : [11],
        '2' : [2],
        '3' : [3],
        '4' : [4],
        '5' : [5],
        '6' : [6],
        '7' : [7],
        '8' : [8],
        '9' : [9],
        '10' : [10],
        'J' : [10],
        'Q' : [10],
        'K' : [10]
    }

    decks = []
    for i in range(number_of_decks):
        for j in range(len(cards)):
            input = cards[j]
            decks.append(input)
            decks.append(input)
            decks.append(input)
            decks.append(input)

    dealer_face = cards.copy() 

    player_cards = []
    for i in range(len(cards)):
        for j in range(len(cards)):
            if j < i:
                continue
            else:
                player_cards.append([cards[i],cards[j]])
    # ------------------------------------------------
    #setting up results table 

    move = ['stand', 'hit']

    if double_allowed == True:
        move.append('double')
    if split_allowed == True:
        move.append('split')
    if surrender_allowed == True:
        move.append('surrender')
    if insurance_allowed == True:
        move.append('insurance')

    results = []
    for i in dealer_face:
        for j in player_cards:
            for k in move:
                results.append({
                    'dealer_face': i, 
                    'player_cards': ','.join(sorted(j)),
                    'move': k,
                    'wins': 0,
                    'losses': 0,
                    'total': 0
                    })

    results_df = pd.DataFrame(results)


    # ------------------------------------------------
    #moves    
    def deal_cards(current_card):
        player_cards = [decks[current_card], decks[current_card + 2]]
        dealer_cards = [decks[current_card + 1], decks[current_card + 3]]
        current_card = current_card + 4
        return player_cards, dealer_cards, current_card
    
    def next_move(first_move, current_card, card_list):
        if first_move == 'Y':
            #add in other moves here
            move = random.choice(['hit', 'stand'])
            if move == 'hit':
                return hit(current_card, card_list)
            else: 
                return stand(card_list, current_card)
        else:
            move = random.choice(['hit', 'stand'])
            if move == 'hit':
                return hit(current_card, card_list)
            else: 
                return stand(card_list, current_card)
    
    def hit(current_card, card_list):
        card_list.append(decks[current_card])
        current_card += 1
        if hand_values(card_list) > 21:
            return 'Bust', current_card
        else:
            return next_move('N', current_card, card_list)
    
    def stand(card_list, current_card):
        return hand_values(card_list), current_card
    
    def dealer_move(dealer_cards, current_card):

        while True:
            value = hand_values(dealer_cards)

            if value > 21:
                return 'Bust', current_card

            if value >= 17:   # TODO: adjust soft 17 later
                return value, current_card

            # Hit
            dealer_cards.append(decks[current_card])
            current_card += 1
         
    # this needs to deal with aces
    def hand_values(card_list):
        output = 0
        for i in card_list:
            output += card_values[i][0]
        return output
    
    # ------------------------------------------------
    #running simulation
    for i in range(sims):
        
        #burn card - shouldn't have any impact
        current_card = 1
        random.shuffle(decks)
        while current_card < round(len(decks)*deck_pen):
            first_move = 'Y'
            player_cards, dealer_cards, current_card = deal_cards(current_card)
            player_result, current_card = next_move(first_move, current_card, player_cards.copy())

            if player_result == 'Bust':
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'losses' ] += 1
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'total' ] += 1
            else:
                dealer_result, current_card = dealer_move(dealer_cards.copy(), current_card)
                if dealer_result == 'Bust':
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'wins' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'total' ] += 1
                elif dealer_result > player_result:
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'losses' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'total' ] += 1
                else:
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'wins' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'total' ] += 1

    return results_df


In [ ]:
output = game_sim()

In [9]:
output[output['player_move_total'] > 0]

,dealer_face,player_cards,player_move_loss,player_move_won,player_move_total
11,A,"A,Q",1571,1499,3070
12,A,"A,K",1051,1017,2068
32,A,"10,3",874,0,874
174,2,"10,Q",1395,1319,2714
185,3,"4,A",446,466,912
204,3,"2,J",10000,0,10000
229,3,"5,6",4322,0,4322
230,3,"5,7",3100,1881,4981
307,4,"3,Q",2204,2280,4484
322,4,"5,8",314,318,632
